# 11. Paoay: calculate crop-model features

**One task:** calculate the saved model’s exact, ordered feature inputs. Reuse the training functions for spectral indices, SAVI and temporal shape. No fitting, classification or accuracy evaluation occurs here.

Keep all eligible Sentinel-2 pixels regardless of Dynamic World guidance. A fixed four-month calendar is not verified crop-cycle alignment. See the [workflow guide](../../docs/paoay_workflow.md).


## 1. Setup and model package

Finish notebooks 06 and 08–10. One model package is selected automatically only when it is the sole candidate. If several exist, set `model_manifest` explicitly to the manifest printed by notebook 06. Nothing is selected by local scores or file age.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import features
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

model_manifest = None  # Optional explicit path to notebook 06's model_manifest.json.
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
guidance_pointer = paoay_root / "land_cover/land_cover_stage.json"
feature_root = paoay_root / "features"
feature_pointer = feature_root / "feature_stage.json"
write_stage_pointer(feature_pointer, status="pending", reason="Notebook 11 has not finished the current feature preparation")
stages = ["Verify model and imagery recipe", "Calculate exact model inputs", "Review coverage and exclusions", "Verify feature handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "11_paoay_features", stages, report_folder.parent / "progress", {"model": str(model_manifest)},
)


## 2. Verify the recipe and inputs

Verify model/recipe checksums, training feature-code provenance, selected column order and imagery settings. The executable model file is not loaded here. Notebook 10’s handoff must reference the same verified Sentinel-2 inputs.


In [ ]:
pipeline_progress.begin_stage("Verify model and imagery recipe")
model_path = features.find_model(report_folder.parent / "analysis", local_path(model_manifest) if model_manifest else None)
guidance_handoff = resolve_stage_pointer(guidance_pointer)
request = features.prepare_request(guidance_handoff, model_path, feature_root)
recipe = features.load_model_recipe(model_path)["recipe"]
print("Selected model:", model_path)
print("Feature design:", recipe["selected_design"])
print("Ordered predictors:", len(recipe["feature_columns"]))
print("Minimum composite clear coverage:", f"{recipe['quality']['minimum_valid_fraction']:.0%}")
show_table([{"position": index, "feature": name} for index, name in enumerate(recipe["feature_columns"], start=1) if index <= 6 or index > len(recipe["feature_columns"]) - 6], title="Predictor order preview (first/last six; full schema saved in handoff)")
pipeline_progress.end_stage()


## 3. Calculate exact features

Mask raw nodata before scaling six spectral bands by 0.0001; never scale clear counts. Require all eight complete bins and finite indices/features; no gap filling or imputation. SAVI and shape features use the saved policy and actual calendar-bin midpoints.

The saved clear-coverage threshold applies separately to each composite. For this mapping demonstration its footprint is the whole study ROI instead of a training parcel. If any bin fails, no pixels advance. This is not permission to keep seven of eight bins.


In [ ]:
pipeline_progress.begin_stage("Calculate exact model inputs")
def pixel_progress(items):
    return pipeline_progress.track(items, "Calculate crop-model features", unit="pixel")
feature_handoff = features.build(request, iterator=pixel_progress)
bundle = features.load_feature_bundle(feature_handoff, require_ready=False)
pipeline_progress.end_stage()


## 4. Review coverage and exclusions

The audit separates missing observations, undefined indices, nonfinite features and failed composite coverage. A zero-row bundle preserves the diagnosis and leaves the stage pending for inspection; downstream prediction cannot silently reuse an older period.


In [ ]:
pipeline_progress.begin_stage("Review coverage and exclusions")
show_table(bundle["coverage"], title="Composite coverage inside the ROI")
show_table(bundle["audit"], title="Pixel readiness and exclusions")
print("ROI pixels:", bundle["manifest"]["roi_pixels"])
print("Ready feature rows:", bundle["manifest"]["row_count"])
print("Matrix shape:", bundle["matrix"].shape)
print("Land-cover filtering applied: no")
pipeline_progress.end_stage()


## 5. Verify and publish the handoff

The compressed matrix preserves the exact saved predictor order. Row/column locations, pixel-centre coordinates and stable IDs allow predictions to be mapped later. Reruns reuse checksum-verified bundles; changed inputs produce separate bundles.


In [ ]:
pipeline_progress.begin_stage("Verify feature handoff")
if bundle["manifest"]["status"] == "prepared":
    features.load_feature_bundle(feature_handoff)
    write_stage_pointer(feature_pointer, feature_handoff)
    print("Feature stage: ready for the saved model")
else:
    write_stage_pointer(feature_pointer, status="pending", reason="No ready feature pixels; inspect notebook 11 coverage and pixel audit")
    print("Feature stage: pending; no usable feature rows")
print("Feature handoff and audit:", feature_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

Numerical feature compatibility does not verify local crop timing or accuracy. Keep the model fixed and report the fixed-calendar limitation alongside eventual predictions.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Current feature pointer:", feature_pointer)
print("No training, crop labels or classification outputs were created.")
if bundle["manifest"]["status"] == "prepared":
    print("Next: notebook 12 will apply the fixed model in a separate batch.")
else:
    print("Next: inspect coverage and exclusion tables before preparing predictions.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
